In [1]:
# src/tune_hyperparameters.py
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import joblib

print("--- Starting Hyperparameter Tuning for XGBoost Classifier ---")

--- Starting Hyperparameter Tuning for XGBoost Classifier ---


In [2]:
# Load the 2018-present dataset
df = pd.read_parquet('../data/processed/hyp_a_features_advanced.parquet')
X = df.drop(columns=['london_direction', 'london_return'])
y = df['london_direction']

In [3]:
X.columns

Index(['day_of_week', 'asia_return', 'asia_range', 'RSI_14', 'MOM_10',
       'STOCHk_14_3_3', 'STOCHd_14_3_3', 'STOCHh_14_3_3', 'CCI_14_0.015',
       'ROC_10', 'CMO_14', 'STOCHRSIk_14_14_3_3', 'STOCHRSId_14_14_3_3',
       'WILLR_14', 'ADX_14', 'ADXR_14_2', 'PSARaf_0.02_0.2', 'PSARr_0.02_0.2',
       'TEMA_10', 'TRIMA_10', 'WMA_10', 'DEMA_10', 'MFI_14', 'BOP', 'ATRr_14',
       'PSAR', 'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9', 'TRIX_30_9',
       'TRIXs_30_9'],
      dtype='object')

In [4]:
# Define the parameter grid
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.001, 0.01, 0.1, 0.2],
    'n_estimators': [100, 200, 500, 1000],
    'subsample': [0.8, 1.0] # Fraction of samples used for fitting each tree
}

In [5]:
# Use TimeSeriesSplit for cross-validation
n_splits = 5 # Use 5 splits
tscv = TimeSeriesSplit(n_splits=n_splits)

# Initialize the XGBoost model
xgb_model = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', use_label_encoder=False, random_state=42)

In [6]:
# Set up GridSearchCV
# We will score based on 'f1' score, which is a good balance of precision and recall.
# n_jobs=-1 uses all available CPU cores to speed it up.
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, 
                           scoring='f1', n_jobs=-1, cv=tscv, verbose=2)

# Save the column order that the model will be trained on
model_columns = list(X.columns)
joblib.dump(model_columns, '../models/xgb_classifier_hyp_a_TUNED_paper_v3_final_model_columns.joblib')
print("Model training columns saved.")

print("Running GridSearchCV... This may take a while.")
grid_search.fit(X, y)

print("\n--- Tuning Complete ---")
print(f"Best F1 Score found: {grid_search.best_score_:.4f}")
print("Best Hyperparameters:")
print(grid_search.best_params_)

Model training columns saved.
Running GridSearchCV... This may take a while.
Fitting 5 folds for each of 96 candidates, totalling 480 fits

--- Tuning Complete ---
Best F1 Score found: 0.5453
Best Hyperparameters:
{'learning_rate': 0.001, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}


c:\Users\mecha\anaconda3\envs\trade_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [06:42:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [7]:
# Save the best model found by the grid search
best_model = grid_search.best_estimator_
joblib.dump(best_model, '../models/xgb_classifier_hyp_a_TUNED_paper_v3_final.joblib')
print("\nBest model saved to 'models/xgb_classifier_hyp_a_TUNED.joblib'")


Best model saved to 'models/xgb_classifier_hyp_a_TUNED.joblib'
